# 🛠️ Лабораторная работа 10. AI-агенты и Tools

Цель: построить минимальную архитектуру агента с Tool Registry, Dispatcher и Agent Loop.

В этой лаборатории решения модели сначала симулируются структурированными Tool Calls. Благодаря этому можно понять механику без внешнего API.


# 1. Импорт

In [ ]:
from typing import Any, Callable

MAX_STEPS = 5

# 2. Создаём Tools

In [ ]:
def add_numbers(a: float, b: float) -> dict[str, Any]:
    return {
        "status": "ok",
        "result": a + b,
    }


def word_count(text: str) -> dict[str, Any]:
    return {
        "status": "ok",
        "result": len(text.split()),
    }


NOTES = {
    "intro": "Искусственный интеллект использует модели и данные.",
    "agent": "Агент может выбирать инструменты и использовать их результаты.",
}


def get_note(name: str) -> dict[str, Any]:
    if name not in NOTES:
        return {
            "status": "error",
            "message": f"Заметка '{name}' не найдена.",
        }

    return {
        "status": "ok",
        "result": NOTES[name],
    }

# 3. Tool Registry

In [ ]:
TOOL_REGISTRY: dict[str, Callable[..., dict[str, Any]]] = {
    "add_numbers": add_numbers,
    "word_count": word_count,
    "get_note": get_note,
}

print(TOOL_REGISTRY.keys())

# 4. Tool Schemas

In [ ]:
TOOL_SCHEMAS = [
    {
        "name": "add_numbers",
        "description": "Складывает два числа a и b.",
        "parameters": {
            "a": "number",
            "b": "number",
        },
    },
    {
        "name": "word_count",
        "description": "Считает количество слов в строке text.",
        "parameters": {
            "text": "string",
        },
    },
    {
        "name": "get_note",
        "description": "Возвращает текст заметки по имени name.",
        "parameters": {
            "name": "string",
        },
    },
]

for schema in TOOL_SCHEMAS:
    print(schema)

# 5. Dispatcher

In [ ]:
def execute_tool(
    tool_name: str,
    arguments: dict[str, Any],
) -> dict[str, Any]:
    if tool_name not in TOOL_REGISTRY:
        return {
            "status": "error",
            "message": f"Неизвестный Tool: {tool_name}",
        }

    tool = TOOL_REGISTRY[tool_name]

    try:
        return tool(**arguments)

    except TypeError as error:
        return {
            "status": "error",
            "message": f"Ошибка аргументов: {error}",
        }

    except Exception as error:
        return {
            "status": "error",
            "message": f"Ошибка Tool: {error}",
        }

# 6. Выполняем Tool Call

In [ ]:
tool_call = {
    "tool_name": "add_numbers",
    "arguments": {
        "a": 10,
        "b": 20,
    },
}

result = execute_tool(
    tool_call["tool_name"],
    tool_call["arguments"],
)

print(result)

# 7. Неизвестный Tool

In [ ]:
result = execute_tool(
    "unknown_tool",
    {},
)

print(result)

# 8. Неверные arguments

In [ ]:
result = execute_tool(
    "add_numbers",
    {
        "a": 10,
    },
)

print(result)

# 9. Tool чтения заметки

In [ ]:
result = execute_tool(
    "get_note",
    {
        "name": "agent",
    },
)

print(result)

# 10. Многошаговая задача

In [ ]:
note_result = execute_tool(
    "get_note",
    {
        "name": "agent",
    },
)

print("Шаг 1:", note_result)

if note_result["status"] == "ok":
    count_result = execute_tool(
        "word_count",
        {
            "text": note_result["result"],
        },
    )

    print("Шаг 2:", count_result)

Здесь агенту понадобилось два инструмента:

```text
get_note
↓
Observation
↓
word_count
↓
Observation
```


# 11. Симулируем решение модели

In [ ]:
def simulated_model(
    user_message: str,
    observations: list[dict[str, Any]],
) -> dict[str, Any]:
    message = user_message.lower()

    if "сколько слов" in message:
        if not observations:
            return {
                "type": "tool_call",
                "tool_name": "get_note",
                "arguments": {
                    "name": "agent",
                },
            }

        last = observations[-1]

        if (
            last["tool_name"] == "get_note"
            and last["result"]["status"] == "ok"
        ):
            return {
                "type": "tool_call",
                "tool_name": "word_count",
                "arguments": {
                    "text": last["result"]["result"],
                },
            }

        if last["tool_name"] == "word_count":
            return {
                "type": "final",
                "content": (
                    "В заметке "
                    f"{last['result']['result']} слов."
                ),
            }

    return {
        "type": "final",
        "content": "Для этой задачи Tool не нужен.",
    }

# 12. Agent Loop

In [ ]:
def run_agent(
    user_message: str,
    max_steps: int = MAX_STEPS,
) -> str:
    observations: list[dict[str, Any]] = []

    for step in range(max_steps):
        model_output = simulated_model(
            user_message,
            observations,
        )

        print(
            f"STEP {step + 1}:",
            model_output,
        )

        if model_output["type"] == "final":
            return model_output["content"]

        if model_output["type"] == "tool_call":
            tool_result = execute_tool(
                model_output["tool_name"],
                model_output["arguments"],
            )

            observation = {
                "tool_name": model_output["tool_name"],
                "result": tool_result,
            }

            observations.append(observation)

            print("OBSERVATION:", observation)

    return "Agent остановлен: достигнут max_steps."


answer = run_agent(
    "Сколько слов в заметке agent?"
)

print("\nFINAL:")
print(answer)

# 13. Ограничение числа шагов

In [ ]:
print(
    run_agent(
        "Сколько слов в заметке agent?",
        max_steps=1,
    )
)

# 14. Read-only и Write Tools

In [ ]:
READ_ONLY_TOOLS = {
    "add_numbers",
    "word_count",
    "get_note",
}

WRITE_TOOLS = {
    "save_file",
    "delete_file",
    "send_message",
}

print("Read-only:", READ_ONLY_TOOLS)
print("Write:", WRITE_TOOLS)

Write Tools в настоящем агенте требуют более строгого контроля, а для важных действий может понадобиться подтверждение пользователя.


# 15. Пример проверки разрешения

In [ ]:
def is_tool_allowed(
    tool_name: str,
    allow_write: bool = False,
) -> bool:
    if tool_name in READ_ONLY_TOOLS:
        return True

    if tool_name in WRITE_TOOLS:
        return allow_write

    return False


print(is_tool_allowed("get_note"))
print(is_tool_allowed("delete_file"))
print(is_tool_allowed("delete_file", allow_write=True))

# 16. 📌 Что нужно запомнить

```text
Model
↓
Tool Call
↓
Dispatcher
↓
Tool Registry
↓
Python Function
↓
Observation
↓
Model
```

LLM выбирает Tool, но Python-код реально выполняет действие.


# 17. 🧩 Эксперименты

Попробуй:

- добавить `multiply_numbers`;
- добавить Tool для поиска максимума в списке;
- сделать Schema для нового Tool;
- обработать неправильный тип аргумента;
- добавить logging каждого Tool Call;
- создать разрешения read/write;
- добавить третий шаг в Agent Loop.


# 18. ➡️ Следующая глава

# Глава 11. RAG — свои документы и знания
